# Doohan sessions to flat and hierarchical LMDPs

This notebook loads a configurable set of sessions from one Doohan maze, reduces every navigation trial to entered maze towers, and scores the resulting independent trajectories under flat and hierarchical goal-conditioned LMDPs.

## 1. Setup paths, select sessions, and build the shared maze

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Run this notebook from the project or notebook directory")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from andrew_mlmdp import (  # noqa: E402
    DoohanMovementDataset,
    LMDPEnvironment,
    NMFDiscoveryParameters,
    SubgoalBasis,
    discover_soft_subgoals,
    plotting as viz,
    score_flat_movement_dataset,
    score_hierarchical_movement_dataset,
    soft_hierarchy_parameters,
)

GRIDMAZE_DATA = (
    PROJECT_ROOT / "external" / "GridMaze-mFC-ephys-DATA" / "data"
)

SUBJECT_IDS = ["m2"]
MAZE_NAME = "maze_1"
START_DATE = "2022-06-23"
END_DATE = "2022-06-25"

movement_dataset = DoohanMovementDataset.from_data_root(
    GRIDMAZE_DATA,
    subject_ids=SUBJECT_IDS,
    start_date=START_DATE,
    end_date=END_DATE,
    maze_name=MAZE_NAME,
)
sessions = movement_dataset.sessions
definition = movement_dataset.definition
maze = definition.maze
environment = LMDPEnvironment(maze)

print(f"sessions: {len(sessions)}")
print(f"maze: {movement_dataset.maze_name}")
print(f"tower grid shape: {maze.shape}")
print(f"physical states: {len(maze.free_cells)}")

## 2. Inspect the assembled navigation trials

The Doohan movement dataset owns session discovery and trial extraction. It removes missing positions and bridge labels, collapses consecutive towers, converts labels to coordinates, and retains malformed sessions or trials as explicit exclusions.

In [ ]:
movement_trials = list(movement_dataset.trials)
exclusion_rows = [
    {
        "session": exclusion.session_id,
        "trial": exclusion.trial_id,
        "goal": exclusion.goal_label,
        "transitions": 0,
        "flat_log_likelihood": float("nan"),
        "hierarchical_log_likelihood": float("nan"),
        "status": "excluded",
        "exclusion_reason": f"data: {exclusion.reason}",
    }
    for exclusion in movement_dataset.exclusions
]

print(f"valid trials: {len(movement_trials)}")
print(f"data exclusions: {len(exclusion_rows)}")

## 3. Inspect the shared maze and one example trial

In [ ]:
tower_labels = dict(definition.label_by_coordinate)
ax = viz.plot_maze(
    maze,
    labels=tower_labels,
    title=f"{MAZE_NAME}: {len(sessions)} selected sessions",
)
if movement_trials:
    example_trial = movement_trials[0]
    example_start = example_trial.trajectory[0]
    ax.plot(
        example_start[1],
        example_start[0],
        marker="o",
        color="#4c956c",
        markersize=9,
        label=f"start ({definition.label_for(example_start)})",
    )
    ax.plot(
        example_trial.goal[1],
        example_trial.goal[0],
        marker="*",
        color="#d1495b",
        markersize=13,
        label=f"goal ({definition.label_for(example_trial.goal)})",
    )
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.show()

## 4. Build one flat environment and one hierarchy template

The distributed basis is discovered once for the maze, not once per session or trial. Goal-conditioned flat solutions and hierarchy tasks are then created lazily and cached by the dataset scorers.

In [ ]:
discovery_rank = 8
smoothness_strength = 1.0
soft_discovery = discover_soft_subgoals(
    environment,
    ranks=(discovery_rank,),
    parameters=NMFDiscoveryParameters(
        lambda_smooth=smoothness_strength,
    ),
    seed=0,
).result(discovery_rank)
viz.plot_soft_subtasks(soft_discovery)
plt.show()

soft_basis = SubgoalBasis.from_profiles(
    maze,
    soft_discovery.profiles,
    core_threshold=0.8,
)
hierarchy_template = environment.hierarchy(
    soft_basis,
    parameters=soft_hierarchy_parameters(
        discovery_rank,
        upper_control_cost=0.18,
    ),
    include_goal_component_while_active=False,
)

## 5. Generate one report per model

Each report represents one model result on the selected dataset. The flat and hierarchical reports remain independent; this notebook displays only the hierarchical dataset summary.

In [ ]:
flat_report = movement_dataset.report(
    score_flat_movement_dataset(
        environment,
        movement_trials,
    )
)
hierarchical_report = movement_dataset.report(
    score_hierarchical_movement_dataset(
        hierarchy_template,
        movement_trials,
    )
)

display(hierarchical_report.summary_dataframe())

example_trial = movement_trials[0] if movement_trials else None
if example_trial is not None:
    example_solution = environment.solve_flat(example_trial.goal)
    goal_state = maze.state_index(example_trial.goal)
    relative_desirability = (
        example_solution.desirability
        / example_solution.desirability[goal_state]
    )
    positive = relative_desirability[relative_desirability > 0.0]
    rows = [coordinate[0] for coordinate in maze.free_cells]
    columns = [coordinate[1] for coordinate in maze.free_cells]

    fig, ax = plt.subplots(figsize=(8, 7))
    viz.plot_maze(maze, show_grid=False, title=None, ax=ax)
    image = ax.scatter(
        columns,
        rows,
        c=relative_desirability,
        cmap="viridis",
        norm=LogNorm(vmin=positive.min(), vmax=1.0),
        s=280,
        edgecolor="white",
        linewidth=0.7,
        zorder=2,
    )
    viz.plot_trajectory_overlay(
        maze,
        example_trial.trajectory,
        goal=example_trial.goal,
        ax=ax,
    )
    ax.legend(loc="upper left", framealpha=0.9)
    ax.set_title(
        "Flat-LMDP and behavior for "
        f"{example_trial.session_id}, trial {example_trial.trial_id}"
    )
    fig.colorbar(image, ax=ax, label="relative desirability (log scale)")
    plt.show()